In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:

import pandas as pd
import numpy as np
import csv
import pickle as pkl
import pickle
import copy
import re
import random
import matplotlib.pyplot as plt
import itertools
import json
import openai
import time
import os


In [ ]:
!rm -rf LLM4BEAR
!git clone --depth 1 --filter=blob:none --sparse https://github.com/anon5159753/LLM4BEAR.git
!cd LLM4BEAR && git sparse-checkout set "BundleRec Data"
!cd LLM4BEAR && git sparse-checkout add "3_Human Evaluation"

In [69]:
def flag_guys(flags):

    l = len(flags[0])

    first_flags = flags[0]
    last_flags = flags[-1]

    regular_flags = [True for _ in range(l)]
    for i in range(l):
        if first_flags[i] == False:
            regular_flags[i] = False
        if last_flags[i] == True:
            regular_flags[i] = False


    empty = []

    for i in range(l):
        if regular_flags[i] == True:
            empty.append(i)


    return empty


In [70]:
domains = ["electronic", "clothing", "food"]

with open(f"/content/LLM4BEAR/3_Human Evaluation/historic_bundle_refinement/historical_bundle_changes_electronic_complete_electronic_no_graph_help_run.pkl", "rb") as f:

    electronic_intents, electronic_bundle_items, electronic_bundle_indices, electronic_scores, electronic_min_scores, electronic_flags = pickle.load(f)

electronic_mod_indices = flag_guys(electronic_flags)


with open(f"/content/LLM4BEAR/3_Human Evaluation/historic_bundle_refinement/historical_bundle_changes_clothing_complete_clothing_no_graph_help_run.pkl", "rb") as f:

    clothing_intents, clothing_bundle_items, clothing_bundle_indices, clothing_scores, clothing_min_scores, clothing_flags = pickle.load(f)


clothing_mod_indices = flag_guys(clothing_flags)

with open(f"/content/LLM4BEAR/3_Human Evaluation/historic_bundle_refinement/historical_bundle_changes_food_complete_food_no_graph_help_run.pkl", "rb") as f:
    food_intents, food_bundle_items, food_bundle_indices, food_scores, food_min_scores, food_flags = pickle.load(f)

food_mod_indices = flag_guys(food_flags)


In [71]:
changed_electronic_scores = copy.deepcopy(electronic_scores)
changed_clothing_scores = copy.deepcopy(clothing_scores)
changed_food_scores = copy.deepcopy(food_scores)

# Basically, if the bundle is size 0, or 1, it is an invalid bundle, therefore you must use the best 2 or more item bundle for the dataset.

In [72]:
for i in range(21):
    for j in range(len(electronic_bundle_items[0])):
        if len(electronic_bundle_items[i][j]) == 1 or len(electronic_bundle_items[i][j]) == 0:
            changed_electronic_scores[i][j] = 0

for i in range(21):
    for j in range(len(clothing_bundle_items[0])):
        if len(clothing_bundle_items[i][j]) == 1 or len(clothing_bundle_items[i][j]) == 0:
            changed_clothing_scores[i][j] = 0

for i in range(21):
    for j in range(len(food_bundle_items[0])) or len(food_bundle_items[i][j]) == 0:
        if len(food_bundle_items[i][j]) == 1:
            changed_food_scores[i][j] = 0

In [73]:
success_electronic = []
unsuccess_electronic = []

last_flags_electronic = electronic_flags[-1]
for i in range(len(last_flags_electronic)):
    if last_flags_electronic[i] == False:
        success_electronic.append(i)
    else:
        unsuccess_electronic.append(i)


best_index_electronic = []

for i in unsuccess_electronic:
    score = []
    for j in range(21):
        score.append(changed_electronic_scores[j][i])
    # print(score)
    best_index_electronic.append(score.index(max(score)))


pseudo_refined_electronic_bundles = []

for i in range(len(unsuccess_electronic)):
    pseudo_refined_electronic_bundles.append(electronic_bundle_items[best_index_electronic[i]][unsuccess_electronic[i]])

pseudo_refined_electronic_intents = []
for i in range(len(unsuccess_electronic)):
    pseudo_refined_electronic_intents.append(electronic_intents[best_index_electronic[i]][unsuccess_electronic[i]])


refined_electronic_bundles = []

for i in range(len(success_electronic)):
    refined_electronic_bundles.append(electronic_bundle_items[-1][success_electronic[i]])

refined_electronic_intents = []

for i in range(len(success_electronic)):
    refined_electronic_intents.append(electronic_intents[-1][success_electronic[i]])


llm4bear_electronic_bundles = [[] for _ in range(len(success_electronic) + len(unsuccess_electronic))]

for i in range(len(success_electronic)):
    llm4bear_electronic_bundles[success_electronic[i]] = refined_electronic_bundles[i]

for i in range(len(unsuccess_electronic)):
    llm4bear_electronic_bundles[unsuccess_electronic[i]] = pseudo_refined_electronic_bundles[i]

llm4bear_electronic_intents = [[] for _ in range(len(success_electronic) + len(unsuccess_electronic))]

for i in range(len(success_electronic)):
    llm4bear_electronic_intents[success_electronic[i]] = refined_electronic_intents[i]

for i in range(len(unsuccess_electronic)):
    llm4bear_electronic_intents[unsuccess_electronic[i]] = pseudo_refined_electronic_intents[i]

In [74]:

success_clothing = []
unsuccess_clothing = []

last_flags_clothing = clothing_flags[-1]
for i in range(len(last_flags_clothing)):
    if last_flags_clothing[i] == False:
        success_clothing.append(i)
    else:
        unsuccess_clothing.append(i)


best_index_clothing = []

for i in unsuccess_clothing:
    score = []
    for j in range(21):
        score.append(changed_clothing_scores[j][i])
    # print(score)
    best_index_clothing.append(score.index(max(score)))


pseudo_refined_clothing_bundles = []

for i in range(len(unsuccess_clothing)):
    pseudo_refined_clothing_bundles.append(clothing_bundle_items[best_index_clothing[i]][unsuccess_clothing[i]])

pseudo_refined_clothing_intents = []
for i in range(len(unsuccess_clothing)):
    pseudo_refined_clothing_intents.append(clothing_intents[best_index_clothing[i]][unsuccess_clothing[i]])


refined_clothing_bundles = []

for i in range(len(success_clothing)):
    refined_clothing_bundles.append(clothing_bundle_items[-1][success_clothing[i]])

refined_clothing_intents = []

for i in range(len(success_clothing)):
    refined_clothing_intents.append(clothing_intents[-1][success_clothing[i]])


llm4bear_clothing_bundles = [[] for _ in range(len(success_clothing) + len(unsuccess_clothing))]

for i in range(len(success_clothing)):
    llm4bear_clothing_bundles[success_clothing[i]] = refined_clothing_bundles[i]

for i in range(len(unsuccess_clothing)):
    llm4bear_clothing_bundles[unsuccess_clothing[i]] = pseudo_refined_clothing_bundles[i]

llm4bear_clothing_intents = [[] for _ in range(len(success_clothing) + len(unsuccess_clothing))]

for i in range(len(success_clothing)):
    llm4bear_clothing_intents[success_clothing[i]] = refined_clothing_intents[i]

for i in range(len(unsuccess_clothing)):
    llm4bear_clothing_intents[unsuccess_clothing[i]] = pseudo_refined_clothing_intents[i]

In [75]:

success_food = []
unsuccess_food = []

last_flags_food = food_flags[-1]
for i in range(len(last_flags_food)):
    if last_flags_food[i] == False:
        success_food.append(i)
    else:
        unsuccess_food.append(i)


best_index_food = []

for i in unsuccess_food:
    score = []
    for j in range(21):
        score.append(changed_food_scores[j][i])
    # print(score)
    best_index_food.append(score.index(max(score)))


pseudo_refined_food_bundles = []

for i in range(len(unsuccess_food)):
    pseudo_refined_food_bundles.append(food_bundle_items[best_index_food[i]][unsuccess_food[i]])

pseudo_refined_food_intents = []
for i in range(len(unsuccess_food)):
    pseudo_refined_food_intents.append(food_intents[best_index_food[i]][unsuccess_food[i]])


refined_food_bundles = []

for i in range(len(success_food)):
    refined_food_bundles.append(food_bundle_items[-1][success_food[i]])

refined_food_intents = []

for i in range(len(success_food)):
    refined_food_intents.append(food_intents[-1][success_food[i]])


llm4bear_food_bundles = [[] for _ in range(len(success_food) + len(unsuccess_food))]

for i in range(len(success_food)):
    llm4bear_food_bundles[success_food[i]] = refined_food_bundles[i]

for i in range(len(unsuccess_food)):
    llm4bear_food_bundles[unsuccess_food[i]] = pseudo_refined_food_bundles[i]

llm4bear_food_intents = [[] for _ in range(len(success_food) + len(unsuccess_food))]

for i in range(len(success_food)):
    llm4bear_food_intents[success_food[i]] = refined_food_intents[i]

for i in range(len(unsuccess_food)):
    llm4bear_food_intents[unsuccess_food[i]] = pseudo_refined_food_intents[i]

In [78]:
print(len(llm4bear_electronic_bundles))
print(len(llm4bear_clothing_bundles))
print(len(llm4bear_food_bundles))
print()
print(len(llm4bear_electronic_intents))
print(len(llm4bear_clothing_intents))
print(len(llm4bear_food_intents))

1750
1910
1784

1750
1910
1784


In [ ]:
# # 1. Save the bundles variable
# with open("/content/drive/My Drive/llm4bear_datasets/llm4bear_electronic_bundles.pkl", "wb") as f:
#     pickle.dump(llm4bear_electronic_bundles, f)

# # 2. Save the intents variable
# with open("/content/drive/My Drive/llm4bear_datasets/llm4bear_electronic_intents.pkl", "wb") as f:
#     pickle.dump(llm4bear_electronic_intents, f)

# # 1. Save the bundles variable
# with open("/content/drive/My Drive/llm4bear_datasets/llm4bear_clothing_bundles.pkl", "wb") as f:
#     pickle.dump(llm4bear_clothing_bundles, f)

# # 2. Save the intents variable
# with open("/content/drive/My Drive/llm4bear_datasets/llm4bear_clothing_intents.pkl", "wb") as f:
#     pickle.dump(llm4bear_clothing_intents, f)


# # 1. Save the bundles variable
# with open("/content/drive/My Drive/llm4bear_datasets/llm4bear_food_bundles.pkl", "wb") as f:
#     pickle.dump(llm4bear_food_bundles, f)

# # 2. Save the intents variable
# with open("/content/drive/My Drive/llm4bear_datasets/llm4bear_food_intents.pkl", "wb") as f:
#     pickle.dump(llm4bear_food_intents, f)